In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [3]:
credentials_location = './de-zoom-camp-449005-be2f15fafa88.json'

conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-latest.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)

In [4]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")

In [5]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [6]:
df_green = spark.read.parquet('gs://nytaxidatawarehouseweek3/*')

In [7]:
df_green.count()

20332093

In [21]:
spark.version

'3.3.2'

In [6]:
df = spark.read \
    .option("header", "true") \
    .parquet('gs://pysparktaxiweek5/yellow_tripdata_2024-10.parquet')

In [7]:
df.count()

3833771

In [12]:
df = df.repartition(4)

In [13]:
df.write.parquet('gs://pysparktaxiweek5/2024-10/')

In [8]:
df.createOrReplaceTempView("yellow")

In [12]:
oct15trip = spark.sql("""
SELECT date_trunc('mon', tpep_pickup_datetime) as month,
date_trunc('day', tpep_pickup_datetime) as day,
COUNT(*)
FROM yellow
group by 1, 2
""")

In [2]:
print(pyspark.__version__)

3.2.2


In [13]:
oct15trip.show()

+-------------------+-------------------+--------+
|              month|                day|count(1)|
+-------------------+-------------------+--------+
|2024-10-01 00:00:00|2024-10-03 00:00:00|  111579|
|2024-10-01 00:00:00|2024-10-07 00:00:00|  101231|
|2024-10-01 00:00:00|2024-10-04 00:00:00|  123691|
|2024-10-01 00:00:00|2024-10-10 00:00:00|  146879|
|2024-09-01 00:00:00|2024-09-30 00:00:00|    5247|
|2024-10-01 00:00:00|2024-10-08 00:00:00|  122156|
|2024-10-01 00:00:00|2024-10-05 00:00:00|  127089|
|2024-10-01 00:00:00|2024-10-06 00:00:00|   86988|
|2024-10-01 00:00:00|2024-10-09 00:00:00|  131700|
|2024-10-01 00:00:00|2024-10-24 00:00:00|  142053|
|2024-10-01 00:00:00|2024-10-11 00:00:00|  136386|
|2024-10-01 00:00:00|2024-10-01 00:00:00|  120182|
|2024-11-01 00:00:00|2024-11-14 00:00:00|       1|
|2024-10-01 00:00:00|2024-10-02 00:00:00|  113664|
|2024-10-01 00:00:00|2024-10-18 00:00:00|  142396|
|2024-10-01 00:00:00|2024-10-16 00:00:00|  136092|
|2024-10-01 00:00:00|2024-10-17

In [22]:
df.schema

StructType(List(StructField(VendorID,IntegerType,true),StructField(tpep_pickup_datetime,TimestampType,true),StructField(tpep_dropoff_datetime,TimestampType,true),StructField(passenger_count,LongType,true),StructField(trip_distance,DoubleType,true),StructField(RatecodeID,LongType,true),StructField(store_and_fwd_flag,StringType,true),StructField(PULocationID,IntegerType,true),StructField(DOLocationID,IntegerType,true),StructField(payment_type,LongType,true),StructField(fare_amount,DoubleType,true),StructField(extra,DoubleType,true),StructField(mta_tax,DoubleType,true),StructField(tip_amount,DoubleType,true),StructField(tolls_amount,DoubleType,true),StructField(improvement_surcharge,DoubleType,true),StructField(total_amount,DoubleType,true),StructField(congestion_surcharge,DoubleType,true),StructField(Airport_fee,DoubleType,true)))

In [15]:
longest_trip = spark.sql("""
SELECT (unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600
FROM yellow
order by 1 desc
""")


In [14]:
spark.catalog.listTables()

[Table(name='yellow', database=None, description=None, tableType='TEMPORARY', isTemporary=True)]

In [16]:
longest_trip.show()

+---------------------------------------------------------------------------------------------------------------------------------+
|((unix_timestamp(tpep_dropoff_datetime, yyyy-MM-dd HH:mm:ss) - unix_timestamp(tpep_pickup_datetime, yyyy-MM-dd HH:mm:ss)) / 3600)|
+---------------------------------------------------------------------------------------------------------------------------------+
|                                                                                                               162.61777777777777|
|                                                                                                                          143.325|
|                                                                                                               137.76055555555556|
|                                                                                                               114.83472222222223|
|                                                                           

In [11]:
print(spark)

In [53]:
filtered_df = df_oct.filter((df_oct.tpep_pickup_datetime >= '2024-10-15 00:00:00') & (df_oct.tpep_pickup_datetime < '2024-10-16 00:00:00'))

In [54]:
filtered_df.count()

128632

In [20]:
taxi_zone_df = spark.read \
    .option("header", "true") \
    .csv('gs://pysparktaxiweek5/taxi_zone_lookup.csv')

In [21]:
taxi_zone_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [25]:
df_join = df.join(taxi_zone_df, df["PULocationID"] == taxi_zone_df["LocationID"], how='inner')

In [26]:
df_join.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+---------+--------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|LocationID|  Borough|                Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+----------+---------+--------------------+------------+
|       2| 2024-09-30 

In [28]:
df_join.createOrReplaceTempView("joined_table")

In [31]:
least_frequent_zone = spark.sql("""
    SELECT Zone, count(*)
    FROM joined_table
    GROUP BY Zone
    ORDER BY 2
    WHERE 
""")

In [32]:
least_frequent_zone.show()

+--------------------+--------+
|                Zone|count(1)|
+--------------------+--------+
|Governor's Island...|       1|
|       Arden Heights|       2|
|       Rikers Island|       2|
|         Jamaica Bay|       3|
| Green-Wood Cemetery|       3|
|Charleston/Totten...|       4|
|       Port Richmond|       4|
|   Rossville/Woodrow|       4|
|Eltingville/Annad...|       4|
|       West Brighton|       4|
|        Crotona Park|       6|
|         Great Kills|       6|
|Heartland Village...|       7|
|     Mariners Harbor|       7|
|Saint George/New ...|       9|
|             Oakwood|       9|
|New Dorp/Midland ...|      10|
|       Broad Channel|      10|
|         Westerleigh|      12|
|     Pelham Bay Park|      12|
+--------------------+--------+
only showing top 20 rows

